# Stage 2 report: slice encoder and embedding store

Run this **after `scripts/model/encode_volumes.py`**. It only reads: the run's manifest, the HU cache and
the embedding store. Nothing is computed with the backbone here, and nothing is written.

It answers:
1. **Coverage**: how many selected volumes have embeddings, per split and kernel.
2. **Input**: what the backbone actually sees after its own HU transform (DALE-CT: clip + z-score).
3. **Sanity**: are the embeddings finite, and are their norms stable across volumes?
4. **Structure along z**: neighbouring slices should be similar (cosine near 1), distant ones less so.
5. **Structure across volumes**: a PCA of the per-volume mean embedding, coloured by kernel, split and
   label. This shows whether kernel (a domain gap) dominates the embedding, which is the motivation for LoRA.

See `docs/model/README.md` (how to run stage 2) and `docs/model/architecture.md` (the store format).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# run from the repository root, wherever the notebook was opened
root = Path.cwd().resolve()
while not (root / "configs" / "preprocessing.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

from ct_preprocessing.cache_record import read_cache_fingerprint
from ct_preprocessing.config import load_config

from ct_model.config import load_encoder_config, load_stage2_config
from ct_model.data.volumes import label_names, load_run_manifest, select_volumes
from ct_model.embeddings.store import open_store

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Pick the run and the encoder

`RUN = None` uses the only run on disk. `ENCODER` is a file in `configs/model/encoders/`.

In [ ]:
STAGE2_CONFIG = "configs/model/encode.yaml"
RUN = None          # e.g. "train-sharp"
ENCODER = None      # None = the default in encode.yaml (dale_ct_2s)

stage2 = load_stage2_config(STAGE2_CONFIG)
data_cfg = load_config(stage2.data_config)
enc_cfg = load_encoder_config(ENCODER or stage2.encoder)
run, manifest = load_run_manifest(data_cfg, stage2.selection.source, RUN or stage2.selection.run)
records = select_volumes(manifest, data_cfg.paths.cache_dir, splits=stage2.selection.splits,
                         qc_passed_only=stage2.selection.qc_passed_only, kernel_classes=stage2.selection.kernel_classes)
store = open_store(stage2.encode.embeddings_dir, enc_cfg,
                   hu_cache_fingerprint=read_cache_fingerprint(data_cfg.paths.cache_dir), create=False)
record = store.read_record()
print(f"run {run.name}: {len(records)} selected volumes")
print(f"store {store.dir}: {len(store.volume_ids())} volumes, {store.size_gb():.2f} GB, dim {store.embed_dim}, {store.dtype}")
print("weights:", record.get("provenance", {}).get("weights_repo"), "@", record.get("provenance", {}).get("weights_commit"))

## 2. Coverage per split and kernel

In [ ]:
vols = pd.DataFrame([{"volume_id": r.volume_id, "split": r.split, "kernel_class": r.kernel_class,
                      "n_slices": r.n_slices, "encoded": store.has(r.volume_id, r.n_slices)} for r in records])
coverage = vols.pivot_table(index="split", columns="kernel_class", values="encoded", aggfunc=["sum", "count"], fill_value=0)
display(coverage)
missing = vols[~vols.encoded]
print(f"{len(missing)} selected volumes not encoded yet" + (f", e.g. {missing.volume_id.head(5).tolist()}" if len(missing) else ""))

## 3. What the backbone sees

Three slices of one volume (the 25th, 50th and 75th percentile positions, head to foot). The top row
uses a lung display window; the bottom row shows the backbone's own input after its HU transform.
The histogram should match the model card's normalisation: roughly zero-centred, clipped at both ends.

In [ ]:
import torch
from ct_model.encoders.transforms import InputTransform

VOLUME_ID = None   # None = the first encoded volume
vid = VOLUME_ID or vols[vols.encoded].volume_id.iloc[0]
rec = next(r for r in records if r.volume_id == vid)
hu = np.load(rec.npy_path, mmap_mode="r")
picks = [int(q * (len(hu) - 1)) for q in (0.25, 0.5, 0.75)]
x = InputTransform(enc_cfg.input)(torch.from_numpy(np.array(hu[picks]))).numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8), gridspec_kw={"width_ratios": [1, 1, 1, 1.3]})
for j, z in enumerate(picks):
    axes[0, j].imshow(hu[z], cmap="gray", vmin=-1000, vmax=400); axes[0, j].set_title(f"slice {z}: HU, lung window")
    axes[1, j].imshow(x[j, 0], cmap="gray"); axes[1, j].set_title(f"slice {z}: model input ch 0")
for ax in axes[:, :3].ravel():
    ax.axis("off")
axes[0, 3].hist(np.asarray(hu[picks]).ravel(), bins=100, color="gray"); axes[0, 3].set_title("HU values")
axes[1, 3].hist(x.ravel(), bins=100, color="gray"); axes[1, 3].set_title(f"model input ({enc_cfg.input.transform})")
fig.suptitle(f"{vid} ({rec.kernel_class}, {rec.split})"); plt.tight_layout(); plt.show()
print(f"model input: mean {x.mean():.3f}, std {x.std():.3f}, min {x.min():.3f}, max {x.max():.3f}")

## 4. Embedding sanity: finite values, norms

The norms should be similar across volumes. Outliers are worth opening in section 3 (artefacts, unusual
field of view, a failed crop).

In [ ]:
encoded = [r for r in records if store.has(r.volume_id, r.n_slices)]
rows = []
for r in encoded:
    e = store.load(r.volume_id).astype(np.float32)
    n = np.linalg.norm(e, axis=1)
    rows.append({"volume_id": r.volume_id, "split": r.split, "kernel_class": r.kernel_class, "n_slices": len(e),
                 "finite": bool(np.isfinite(e).all()), "norm_mean": n.mean(), "norm_std": n.std(), "abs_max": np.abs(e).max()})
stats = pd.DataFrame(rows)
print(f"{len(stats)} volumes, all finite: {stats.finite.all()}, largest |x|: {stats.abs_max.max():.2f} (float16 max 65504)")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for kc, g in stats.groupby("kernel_class"):
    ax[0].hist(g.norm_mean, bins=30, alpha=0.6, label=kc)
ax[0].set_title("mean slice-embedding norm per volume"); ax[0].legend()
ax[1].scatter(stats.n_slices, stats.norm_mean, s=8); ax[1].set_xlabel("slices"); ax[1].set_ylabel("mean norm")
plt.tight_layout(); plt.show()
ordered = stats.sort_values("norm_mean")
display(pd.concat([ordered.head(2), ordered.tail(2)]).drop_duplicates("volume_id"))  # the extremes

## 5. Structure along z (one volume)

Left: cosine similarity of each slice with the next. It should stay high, with dips at anatomical
transitions (e.g. neck to apex, lung base to abdomen). Right: the slice-by-slice cosine matrix. A block
structure means the encoder separates anatomical regions, which is what an aggregator can attend over.

In [ ]:
e = store.load(vid).astype(np.float32)
u = e / np.linalg.norm(e, axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot((u[1:] * u[:-1]).sum(1)); ax[0].set_xlabel("slice (head -> foot)"); ax[0].set_ylabel("cosine with next slice")
im = ax[1].imshow(u @ u.T, cmap="viridis"); ax[1].set_title(f"{vid}: slice x slice cosine"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

## 6. Structure across volumes: PCA of mean embeddings

Each point is one volume (the mean of its slice embeddings, the crudest possible aggregation). If points
split mainly by **kernel**, the frozen encoder carries a strong kernel signal: a domain gap for
sharp-trained models tested on soft scans, which phase 2 (LoRA) targets. Two reconstructions of the same
scan landing close together is a good sign.

In [ ]:
LABEL = None   # e.g. "emphysema"; None = the first label column
if len(encoded) < 3:
    raise SystemExit(f"only {len(encoded)} volume(s) encoded -- PCA needs at least 3; encode more first")
means = np.stack([store.load(r.volume_id).astype(np.float32).mean(0) for r in encoded])
centred = means - means.mean(0)
_, s, vt = np.linalg.svd(centred, full_matrices=False)
pc = centred @ vt[:2].T
explained = (s[:2] ** 2) / (s ** 2).sum()

names = label_names(manifest)
label = LABEL or (names[0] if names else None)
fig, axes = plt.subplots(1, 3 if label else 2, figsize=(18, 5))
for ax, key in zip(axes, ["kernel_class", "split"]):
    for value in sorted({getattr(r, key) for r in encoded}):
        sel = np.array([getattr(r, key) == value for r in encoded])
        ax.scatter(pc[sel, 0], pc[sel, 1], s=10, alpha=0.7, label=value or "(none)")
    ax.set_title(f"coloured by {key}"); ax.legend()
if label:
    li = names.index(label)
    y = np.array([r.labels[li] for r in encoded])
    for value, colour in ((0.0, "tab:blue"), (1.0, "tab:red")):
        sel = y == value
        axes[2].scatter(pc[sel, 0], pc[sel, 1], s=10, alpha=0.7, c=colour, label=f"{label}={int(value)}")
    axes[2].set_title(f"coloured by {label}"); axes[2].legend()
for ax in axes:
    ax.set_xlabel(f"PC1 ({explained[0]:.0%})"); ax.set_ylabel(f"PC2 ({explained[1]:.0%})")
plt.tight_layout(); plt.show()

## Next

Stage 3 (ABMIL) and stage 4 (classifier) train on this store; see `docs/model/architecture.md`, "Roadmap".
Their reports (per-label AUROC, per-kernel test results, attention over slices) belong in a separate
notebook that reads the experiment's output folder.